# Mining Agent Observability → Evaluation Datasets

**The Flywheel:** Deploy → Observe → Mine → Evaluate → Improve → Redeploy

This notebook demonstrates how to extract evaluation signal from production agent observability data. We'll mine three types of feedback from `SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS`:

1. **Explicit feedback** — thumbs up/down signals submitted by users
2. **Implicit negative feedback** — detecting when users rephrase the same question (frustration signal)
3. **Intent trends** — classifying user queries to understand usage distribution and coverage gaps

These signals feed directly into an evaluation dataset that tests the agent on its real weak spots.

**Agent under test:** CMO Assistant (Cortex Analyst + Cortex Search)

**Prerequisites:**
- Run `setup.sql` to create the `CMO_EVAL_LAB` environment
- `SNOWFLAKE.CORTEX_USER` database role granted
- `READ UNREDACTED AI OBSERVABILITY EVENTS TABLE` privilege (for feedback text)
- Cross-region inference enabled

---
## Section 1: Setup & Agent Verification

In [1]:
import os
import json
import time

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.configs({
        "connection_name": "parker_demo",
        "warehouse": "CMO_EVAL_WH",
        "database": "CMO_EVAL_LAB",
        "schema": "PUBLIC",
    }).create()

session.sql("USE DATABASE CMO_EVAL_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE CMO_EVAL_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

Connected as: PERICKSON
Role: ACCOUNTADMIN


In [2]:
# Create (or replace) the CMO Assistant agent
session.sql("""
CREATE OR REPLACE AGENT CMO_ASSISTANT
  COMMENT = 'Marketing/Finance assistant for campaign performance and strategy'
FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  response: |
    You are a CMO assistant that helps marketing leaders understand campaign performance,
    budget allocation, and strategic recommendations. Be concise and data-driven.
    When presenting financial data, always include the time period and round to 2 decimal places.
    When asked for a summary, brief, or executive-level view, format the response as:
    - A one-line headline insight
    - 3-5 bullet points with key findings
    - A recommended action
    Keep total length under 200 words.
  orchestration: |
    For quantitative questions about spend, revenue, ROI, conversions, or performance metrics, use the campaign_analytics tool.
    For questions about strategy, methodology, planning documents, or guidelines, use the strategy_search tool.
    For requests to summarize or create executive briefs, first gather data with the appropriate tool, then format as an executive summary.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: campaign_analytics
      description: "Query structured campaign performance data including spend, revenue, impressions, clicks, conversions, ROI, CPC, and CPA by channel and time period."
  - tool_spec:
      type: cortex_search
      name: strategy_search
      description: "Search marketing strategy documents, budget methodology, attribution models, benchmarks, and planning briefs."
tool_resources:
  campaign_analytics:
    semantic_view: CMO_EVAL_LAB.PUBLIC.CMO_ANALYTICS
    execution_environment:
        type: warehouse
        warehouse: CMO_EVAL_WH
  strategy_search:
    name: CMO_EVAL_LAB.PUBLIC.STRATEGY_SEARCH_SVC
    max_results: "3"
$$
""").collect()
print("Agent CMO_ASSISTANT created.")

Agent CMO_ASSISTANT created.


In [3]:
# Quick smoke test
result = session.sql("""
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "What was total spend in 2024?"}]}]}$$
) AS response
""").collect()
resp = json.loads(result[0]['RESPONSE'])
text = next((c.get('text') for c in resp.get('content', []) if c.get('type') == 'text'), '(no response)')
print("Smoke test passed." if text != '(no response)' else "WARNING: No text response.")
print(f"Response preview: {text[:200]}...")

Smoke test passed.
Response preview: Total marketing spend for full-year 2024 was **$1,723,500.00**....


---
## Section 2: Simulate Production Traffic (~100 queries)

We'll send a realistic distribution of queries to the agent, representing what real CMO-facing usage looks like:
- **Analytics queries** (40%) — spend, ROI, CPC, conversions, comparisons
- **Strategy/search queries** (25%) — methodology, planning, guidelines
- **Multi-tool queries** (15%) — questions requiring both data + strategy context
- **Executive summary requests** (10%) — formatted briefs
- **Out-of-scope/edge cases** (10%) — competitor questions, vague asks, irrelevant topics

In [4]:
# Define ~100 queries across intent categories
# Each entry: (query, intent, expected_feedback, feedback_message)
# expected_feedback: 'positive', 'negative', or None (no feedback submitted)
# In production, ~80% of queries receive NO feedback at all.
# Only ~20% of users bother to click thumbs up/down.

SIMULATED_QUERIES = [
    # ═══════════════════════════════════════════════════════════════════
    # ANALYTICS QUERIES (40 queries) — ~5 get feedback
    # ═══════════════════════════════════════════════════════════════════
    ("What was total spend in 2024?", "analytics", None, ""),
    ("What was total revenue in 2024?", "analytics", None, ""),
    ("What was total spend for Paid Search in 2024?", "analytics", None, ""),
    ("Show me total conversions by channel for 2024", "analytics", None, ""),
    ("What was the CPC for Social Media in Q2?", "analytics", None, ""),
    ("Which channel had the highest ROI in 2024?", "analytics", "positive", "Exactly what I needed"),
    ("Compare ROI across all channels for Q4 2024", "analytics", None, ""),
    ("What was the conversion rate for Email in 2024?", "analytics", None, ""),
    ("Show me monthly spend trends for Paid Search", "analytics", None, ""),
    ("What was our ROAS for Display advertising?", "analytics", "negative", "Showed overall ROAS but I asked specifically about Display"),
    ("How much did we spend on the Q4 Holiday Push campaign?", "analytics", None, ""),
    ("What was the CPA for Paid Search in Q1 vs Q4?", "analytics", None, ""),
    ("Total impressions in 2024 across all channels?", "analytics", None, ""),
    ("Which month had the highest revenue?", "analytics", None, ""),
    ("What percentage of total spend went to Social Media?", "analytics", "negative", "Gave me the raw number but not the percentage"),
    ("Show me clicks per channel for H2 2024", "analytics", None, ""),
    ("What was the total spend in Q1 2024?", "analytics", None, ""),
    ("Compare Email and Display ROI", "analytics", None, ""),
    ("Which campaign drove the most conversions?", "analytics", None, ""),
    ("What was our spend efficiency (revenue per dollar spent) by channel?", "analytics", "negative", "Confused ROAS with ROI in the explanation"),
    ("Show me the month-over-month growth in revenue for Paid Search", "analytics", None, ""),
    ("What was total spend in H1 vs H2?", "analytics", None, ""),
    ("How many total clicks did we get in 2024?", "analytics", None, ""),
    ("What was average monthly spend across all channels?", "analytics", None, ""),
    ("Which quarter had the best ROAS?", "analytics", None, ""),
    ("Show me conversion trends for Email over 2024", "analytics", None, ""),
    ("What was the CPC trend for Paid Search across quarters?", "analytics", None, ""),
    ("Total revenue from the Q2 Product Launch campaign?", "analytics", None, ""),
    ("How did Social Media perform in Q4 vs Q1?", "analytics", None, ""),
    ("What was our worst performing channel by ROI?", "analytics", None, ""),
    ("Break down Q4 spend by channel", "analytics", None, ""),
    ("What was the total number of conversions in December?", "analytics", None, ""),
    ("Show me Display impressions by quarter", "analytics", None, ""),
    ("What was our blended CPA across all channels for 2024?", "analytics", None, ""),
    ("Revenue from Email in Q4?", "analytics", None, ""),
    ("Which channel had the lowest CPC?", "analytics", "negative", "Said Email had lowest CPC but Email doesn't really have traditional CPC"),
    ("Total spend on H2 Performance Max campaign", "analytics", None, ""),
    ("Compare Q1 Brand Awareness results across channels", "analytics", None, ""),
    ("What was the click-through rate by channel?", "analytics", None, ""),
    ("Show me the top 3 months by conversion volume", "analytics", "positive", ""),

    # ═══════════════════════════════════════════════════════════════════
    # STRATEGY/SEARCH QUERIES (25 queries) — ~5 get feedback
    # ═══════════════════════════════════════════════════════════════════
    ("What is our attribution methodology?", "strategy", "negative", "Missed the 30-day lookback window detail"),
    ("What are the 2024 performance benchmarks for Paid Search?", "strategy", None, ""),
    ("How do we allocate budget across channels?", "strategy", "positive", "Clear summary"),
    ("What are the brand guidelines for reporting ROI?", "strategy", None, ""),
    ("Tell me about the Q4 planning brief", "strategy", None, ""),
    ("What triggers a budget reallocation?", "strategy", None, ""),
    ("How is first-touch vs last-touch weighted in attribution?", "strategy", "negative", "Only gave a high-level overview, didnt mention the specific 0.2 and 0.3 weights"),
    ("What are our target benchmarks for Social Media?", "strategy", None, ""),
    ("What is the minimum viable spend policy?", "strategy", None, ""),
    ("How should executive summaries be formatted?", "strategy", None, ""),
    ("What does the channel strategy say about flex budget?", "strategy", None, ""),
    ("What is the target ROAS for Display?", "strategy", None, ""),
    ("How often does attribution data refresh?", "strategy", "negative", "Didnt mention the 48-hour lag for conversion window"),
    ("What are the Q4 success metrics?", "strategy", None, ""),
    ("Tell me about our influencer partnership strategy", "strategy", None, ""),
    ("How do we report percentages according to brand guidelines?", "strategy", None, ""),
    ("What is the email frequency plan for Q4?", "strategy", None, ""),
    ("What percentage of budget goes to Paid Search per strategy?", "strategy", None, ""),
    ("When do channels get flagged for increased investment?", "strategy", None, ""),
    ("What is the target CPC range for Paid Search?", "strategy", None, ""),
    ("Describe our multi-touch attribution approach", "strategy", None, ""),
    ("What are the three inputs for budget allocation?", "strategy", None, ""),
    ("What is the planned Q4 budget increase over Q3?", "strategy", None, ""),
    ("How should year-over-year comparisons be presented?", "strategy", None, ""),
    ("What is the target conversion rate for Email?", "strategy", "positive", ""),

    # ═══════════════════════════════════════════════════════════════════
    # MULTI-TOOL QUERIES (15 queries) — ~4 get feedback
    # ═══════════════════════════════════════════════════════════════════
    ("Compare our actual Q4 spend to what the planning brief recommended", "multi_tool", "negative", "Only showed the actual spend, didnt reference the planning brief target"),
    ("Is our Paid Search CPC within the benchmark range?", "multi_tool", "positive", "Good — pulled both actual data and benchmark"),
    ("Did we hit the Q4 ROAS target of 3.5x?", "multi_tool", None, ""),
    ("How does our actual Email conversion rate compare to the 7% target?", "multi_tool", None, ""),
    ("Are there channels exceeding ROI targets by 20% that should get budget increases?", "multi_tool", "negative", "Only looked at data, didnt reference the budget allocation methodology about the 20% rule"),
    ("Did we achieve the 15000 conversion target in Q4?", "multi_tool", None, ""),
    ("Is Social Media meeting its target ROAS of 1.8x?", "multi_tool", None, ""),
    ("Compare actual Display CPM to our $18-22 benchmark", "multi_tool", "negative", "Said it couldnt calculate CPM"),
    ("Which channels are underperforming their ROI targets?", "multi_tool", None, ""),
    ("Is our Q4 CPA staying below the $45 target?", "multi_tool", None, ""),
    ("How does our budget split compare to the recommended 40/25/10/15 allocation?", "multi_tool", None, ""),
    ("Did the H2 Performance Max campaign meet Paid Search benchmarks?", "multi_tool", None, ""),
    ("Are we spending within the planned Q4 budget of $590k?", "multi_tool", None, ""),
    ("Given our attribution methodology, which channel gets most first-touch credit?", "multi_tool", "negative", "Couldnt answer - said it doesnt have first-touch data"),
    ("Summarize how our actual 2024 performance aligns with strategy recommendations", "multi_tool", None, ""),

    # ═══════════════════════════════════════════════════════════════════
    # EXECUTIVE SUMMARY REQUESTS (10 queries) — ~3 get feedback
    # ═══════════════════════════════════════════════════════════════════
    ("Give me an executive summary of 2024 marketing performance", "executive_summary", None, ""),
    ("Create a brief for the board on Q4 results", "executive_summary", "negative", "Too long and not in executive brief format"),
    ("Summarize channel performance for the CMO", "executive_summary", None, ""),
    ("One-pager on our best and worst performing channels", "executive_summary", None, ""),
    ("Executive brief: are we on track for 2024 goals?", "executive_summary", "negative", "Didnt structure as headline + bullets + recommendation"),
    ("Quick summary of Paid Search 2024 performance", "executive_summary", None, ""),
    ("Brief the CFO on marketing ROI by channel", "executive_summary", "positive", "Well formatted"),
    ("Prepare a quarterly review summary for Q4", "executive_summary", None, ""),
    ("Give me the headline metric for 2024 overall", "executive_summary", None, ""),
    ("Summarize our worst-performing area and what to do about it", "executive_summary", None, ""),

    # ═══════════════════════════════════════════════════════════════════
    # OUT-OF-SCOPE / EDGE CASES (10 queries) — ~3 get feedback
    # ═══════════════════════════════════════════════════════════════════
    ("What are our competitors spending on paid search?", "out_of_scope", None, ""),
    ("Can you book a meeting with the CMO?", "out_of_scope", None, ""),
    ("Write me a blog post about digital marketing trends", "out_of_scope", "negative", "Should have declined but tried to write one"),
    ("What is the weather today?", "out_of_scope", None, ""),
    ("Send an email to the marketing team about Q4 results", "out_of_scope", None, ""),
    ("What is Snowflake's stock price?", "out_of_scope", None, ""),
    ("Delete all campaign data from January", "out_of_scope", "positive", "Good that it refused"),
    ("How much should we pay our social media manager?", "out_of_scope", "negative", "Tried to answer using campaign data which is unrelated"),
    ("What did we spend on TV advertising?", "out_of_scope", None, ""),
    ("Tell me a joke about marketing", "out_of_scope", None, ""),
]

print(f"Total queries defined: {len(SIMULATED_QUERIES)}")
print(f"\nIntent distribution:")
from collections import Counter
intent_counts = Counter(q[1] for q in SIMULATED_QUERIES)
for intent, count in intent_counts.most_common():
    print(f"  {intent:20s} {count:3d} ({count/len(SIMULATED_QUERIES)*100:.0f}%)")

feedback_count = sum(1 for q in SIMULATED_QUERIES if q[2] is not None)
print(f"\nFeedback rate: {feedback_count}/{len(SIMULATED_QUERIES)} ({feedback_count/len(SIMULATED_QUERIES)*100:.0f}%) — realistic production rate")


Total queries defined: 100

Intent distribution:
  analytics             40 (40%)
  strategy              25 (25%)
  multi_tool            15 (15%)
  executive_summary     10 (10%)
  out_of_scope          10 (10%)

Feedback rate: 22/100 (22%) — realistic production rate


In [5]:
# Send all queries to the agent via REST API to capture request_ids for feedback
# This simulates real production traffic over time

import requests as req
import os
# Get connection details for REST API
host = session.connection.host
try:
    with open('/snowflake/session/token', 'r') as f:
        token = f.read().strip()
    token_type = "OAUTH"
except FileNotFoundError:
    # Local environment: use PAT from connection config
    import toml
    cfg = toml.load(os.path.expanduser("~/.snowflake/config.toml"))
    token = cfg["connections"]["parker_demo"]["password"]
    token_type = "PROGRAMMATIC_ACCESS_TOKEN"

# Snowflake URLs use hyphens, not underscores
host = host.replace('_', '-')
print(f"Using host: {host}")
print(f"Auth type: {token_type}")

agent_url = f"https://{host}/api/v2/databases/CMO_EVAL_LAB/schemas/PUBLIC/agents/CMO_ASSISTANT:run"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "X-Snowflake-Authorization-Token-Type": token_type
}

results = []
total = len(SIMULATED_QUERIES)

print(f"Sending {total} queries to CMO_ASSISTANT via REST API...")
print("(This will take several minutes)\n")

for i, (query, intent, feedback, msg) in enumerate(SIMULATED_QUERIES):
    try:
        body = {
            "messages": [{"role": "user", "content": [{"type": "text", "text": query}]}],
            "stream": False
        }
        resp = req.post(agent_url, headers=headers, json=body)
        resp.raise_for_status()
        data = resp.json()

        # request_id is in the x-snowflake-request-id response header
        request_id = resp.headers.get('x-snowflake-request-id', '')
        text = next((c.get('text') for c in data.get('content', []) if c.get('type') == 'text'), '')

        results.append({
            'index': i,
            'query': query,
            'intent': intent,
            'feedback': feedback,
            'feedback_msg': msg,
            'request_id': request_id,
            'response_text': text[:500],
            'status': 'ok'
        })

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{total}] completed")

    except Exception as e:
        results.append({
            'index': i,
            'query': query,
            'intent': intent,
            'feedback': feedback,
            'feedback_msg': msg,
            'request_id': '',
            'response_text': '',
            'status': f'error: {str(e)[:100]}'
        })

ok_count = sum(1 for r in results if r['status'] == 'ok')
has_rid = sum(1 for r in results if r['request_id'])
print(f"\nDone. {ok_count}/{total} queries succeeded.")
print(f"Request IDs captured: {has_rid}/{ok_count}")

Using host: SFSENORTHAMERICA-PERICKSON-AWS1.snowflakecomputing.com
Auth type: PROGRAMMATIC_ACCESS_TOKEN
Sending 100 queries to CMO_ASSISTANT via REST API...
(This will take several minutes)

  [10/100] completed
  [20/100] completed
  [30/100] completed
  [40/100] completed
  [50/100] completed
  [60/100] completed
  [70/100] completed
  [80/100] completed
  [90/100] completed
  [100/100] completed

Done. 100/100 queries succeeded.
Request IDs captured: 100/100


---
## Section 3: Submit Explicit Feedback (Thumbs Up/Down)

In production, users click thumbs-up or thumbs-down in the UI (CoWork, custom app). Here we simulate that by calling the Feedback REST API for each interaction where we have a feedback signal.

In [6]:
# Submit feedback for queries that have a feedback signal
feedback_url = f"https://{host}/api/v2/databases/CMO_EVAL_LAB/schemas/PUBLIC/agents/CMO_ASSISTANT:feedback"

feedback_results = []
feedback_queries = [r for r in results if r['feedback'] is not None and r['request_id']]

print(f"Submitting feedback for {len(feedback_queries)} interactions...\n")

pos_count = 0
neg_count = 0

for r in feedback_queries:
    is_positive = r['feedback'] == 'positive'
    body = {
        "orig_request_id": r['request_id'],
        "positive": is_positive,
    }
    if r['feedback_msg']:
        body["feedback_message"] = r['feedback_msg']

    try:
        fb_resp = req.post(feedback_url, headers=headers, json=body)
        status = "OK" if fb_resp.status_code in (200, 201) else f"ERR:{fb_resp.status_code}"
    except Exception as e:
        status = f"ERR:{str(e)[:50]}"

    if is_positive:
        pos_count += 1
    else:
        neg_count += 1

    feedback_results.append({'query': r['query'], 'signal': r['feedback'], 'status': status})

ok_fb = sum(1 for f in feedback_results if f['status'] == 'OK')
print(f"Feedback submitted: {ok_fb}/{len(feedback_queries)} succeeded")
print(f"  Thumbs up:   {pos_count}")
print(f"  Thumbs down: {neg_count}")

# Write a request_id → query → feedback mapping table for reliable joins in observability queries
# (Observability doesn't store orig_request_id on feedback events, so we persist the mapping ourselves)
from snowflake.snowpark.types import StructType, StructField, StringType
session.sql("CREATE OR REPLACE TABLE FEEDBACK_REQUEST_MAPPING (REQUEST_ID VARCHAR, USER_QUERY VARCHAR, FEEDBACK_SIGNAL VARCHAR, FEEDBACK_MESSAGE VARCHAR)").collect()
mapping_rows = [
    (r['request_id'], r['query'], r['feedback'], r['feedback_msg'])
    for r in results if r['feedback'] is not None and r['request_id']
]
mapping_df = session.create_dataframe(mapping_rows, schema=['REQUEST_ID', 'USER_QUERY', 'FEEDBACK_SIGNAL', 'FEEDBACK_MESSAGE'])
mapping_df.write.mode("overwrite").save_as_table("FEEDBACK_REQUEST_MAPPING")
print(f"\nFEEDBACK_REQUEST_MAPPING table created with {len(mapping_rows)} rows (links request_ids to feedback)")

Submitting feedback for 22 interactions...

Feedback submitted: 22/22 succeeded
  Thumbs up:   7
  Thumbs down: 15

FEEDBACK_REQUEST_MAPPING table created with 22 rows (links request_ids to feedback)


---
## Section 4: Simulate Implicit Negative Feedback (Rephrasing)

When a user asks the same question twice (rephrased) within the same thread, it's a strong signal that the first answer was unsatisfactory — even without an explicit thumbs-down.

We simulate this by sending multi-turn conversations where the user rephrases their question.

In [7]:
# Define multi-turn conversations where the user rephrases (implicit dissatisfaction)
REPHRASE_CONVERSATIONS = [
    {
        "thread_label": "rephrase_roi_detail",
        "turns": [
            "What is the ROI for each channel?",
            "I meant the ROI broken down by channel for the full year 2024, can you show each one separately?",
        ]
    },
    {
        "thread_label": "rephrase_attribution_weights",
        "turns": [
            "How does attribution work?",
            "No, I want the specific weights - what percentage does first-touch get vs last-touch vs middle touches?",
        ]
    },
    {
        "thread_label": "rephrase_budget_percentage",
        "turns": [
            "What is Social Media's share of the total budget?",
            "I need it as a percentage of the total annual spend, not the dollar amount",
        ]
    },
    {
        "thread_label": "rephrase_q4_target",
        "turns": [
            "Did we hit our Q4 targets?",
            "Specifically, the Q4 planning brief said ROAS of 3.5x, CPA below $45, and 15000 conversions. Did we achieve each of those three targets?",
        ]
    },
    {
        "thread_label": "rephrase_cpc_trend",
        "turns": [
            "CPC trend",
            "Show me how cost per click changed month over month for Paid Search throughout 2024",
        ]
    },
    {
        "thread_label": "rephrase_exec_format",
        "turns": [
            "Summarize Q4 performance",
            "Format that as an executive brief with a headline, bullet points, and a recommended action - keep it under 200 words",
        ]
    },
    {
        "thread_label": "rephrase_benchmark_comparison",
        "turns": [
            "Are we meeting benchmarks?",
            "Compare our actual CPC, conversion rate, and ROAS for each channel against the 2024 internal benchmarks",
        ]
    },
    {
        "thread_label": "rephrase_display_performance",
        "turns": [
            "How is Display doing?",
            "Give me Display advertising metrics for 2024 - total spend, impressions, clicks, conversions, and ROAS",
        ]
    },
    {
        "thread_label": "rephrase_email_efficiency",
        "turns": [
            "Is Email efficient?",
            "What is the cost per conversion for Email compared to other channels? I want to see why we say Email is the most efficient",
        ]
    },
    {
        "thread_label": "rephrase_yoy_methodology",
        "turns": [
            "How should I present year-over-year data?",
            "What do the brand guidelines specifically say about year-over-year comparisons and methodology changes?",
        ]
    },
]

print(f"Defined {len(REPHRASE_CONVERSATIONS)} rephrase conversations ({sum(len(c['turns']) for c in REPHRASE_CONVERSATIONS)} total turns)")

Defined 10 rephrase conversations (20 total turns)


In [8]:
# Send multi-turn conversations using threads to simulate rephrasing behavior
rephrase_results = []
threads_url = f"https://{host}/api/v2/cortex/threads"

print(f"Sending {len(REPHRASE_CONVERSATIONS)} multi-turn conversations (with threads)...\n")

for conv_idx, conv in enumerate(REPHRASE_CONVERSATIONS):
    conv_request_ids = []

    # Create a new thread for this conversation
    thread_resp = req.post(threads_url, headers=headers, json={})
    thread_resp.raise_for_status()
    thread_id = thread_resp.json()['thread_id']

    parent_message_id = 0  # First message in thread

    for turn_idx, turn_text in enumerate(conv['turns']):
        try:
            body = {
                "thread_id": thread_id,
                "parent_message_id": parent_message_id,
                "messages": [{"role": "user", "content": [{"type": "text", "text": turn_text}]}],
                "stream": False
            }
            resp = req.post(agent_url, headers=headers, json=body)
            resp.raise_for_status()
            data = resp.json()

            request_id = resp.headers.get('x-snowflake-request-id', '')
            # Get assistant_message_id for next turn's parent
            parent_message_id = data.get('metadata', {}).get('assistant_message_id', 0)
            conv_request_ids.append(request_id)

        except Exception as e:
            conv_request_ids.append('')
            # If a turn fails, we can't continue the thread reliably
            break

    rephrase_results.append({
        'label': conv['thread_label'],
        'turns': conv['turns'],
        'request_ids': conv_request_ids,
        'thread_id': thread_id,
    })

    print(f"  [{conv_idx+1}/{len(REPHRASE_CONVERSATIONS)}] {conv['thread_label']} ({len(conv_request_ids)} turns)")

print(f"\nDone. All rephrase conversations sent.")

Sending 10 multi-turn conversations (with threads)...

  [1/10] rephrase_roi_detail (1 turns)
  [2/10] rephrase_attribution_weights (2 turns)
  [3/10] rephrase_budget_percentage (2 turns)
  [4/10] rephrase_q4_target (2 turns)
  [5/10] rephrase_cpc_trend (2 turns)
  [6/10] rephrase_exec_format (2 turns)
  [7/10] rephrase_benchmark_comparison (2 turns)
  [8/10] rephrase_display_performance (2 turns)
  [9/10] rephrase_email_efficiency (2 turns)
  [10/10] rephrase_yoy_methodology (2 turns)

Done. All rephrase conversations sent.


In [9]:
rephrase_results

[{'label': 'rephrase_roi_detail',
  'turns': ['What is the ROI for each channel?',
   'I meant the ROI broken down by channel for the full year 2024, can you show each one separately?'],
  'request_ids': [''],
  'thread_id': 58553664125},
 {'label': 'rephrase_attribution_weights',
  'turns': ['How does attribution work?',
   'No, I want the specific weights - what percentage does first-touch get vs last-touch vs middle touches?'],
  'request_ids': ['b592d4b7-06ff-471a-ad58-eed6959a0bca',
   '101e6e7f-97e0-4f55-9cb1-ab522a9b41d2'],
  'thread_id': 58553664129},
 {'label': 'rephrase_budget_percentage',
  'turns': ["What is Social Media's share of the total budget?",
   'I need it as a percentage of the total annual spend, not the dollar amount'],
  'request_ids': ['e592b032-ba4c-4d5d-ad44-f66b6d241a21',
   '76e77831-f3ca-4d9d-9676-0fd7b8d84865'],
  'thread_id': 58553664133},
 {'label': 'rephrase_q4_target',
  'turns': ['Did we hit our Q4 targets?',
   'Specifically, the Q4 planning brief 

---
## Section 5: Mine Direct Feedback from Observability

Now we query `SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS` to extract the feedback signals we just submitted. In production, this data accumulates organically from real users.

**Key filter:** `RECORD:name = 'CORTEX_AGENT_FEEDBACK'`

In [10]:
# Observability events may take a few seconds to propagate
print("Waiting 15 seconds for observability events to propagate...")
time.sleep(15)
print("Done.")

Waiting 15 seconds for observability events to propagate...
Done.


In [11]:
# Overall feedback summary
session.sql("""
SELECT
    COUNT(*) AS TOTAL_FEEDBACK,
    SUM(CASE WHEN VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) AS THUMBS_UP,
    SUM(CASE WHEN NOT VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) AS THUMBS_DOWN,
    ROUND(SUM(CASE WHEN VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS APPROVAL_RATE_PCT
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
))
WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
""").show()

------------------------------------------------------------------------
|"TOTAL_FEEDBACK"  |"THUMBS_UP"  |"THUMBS_DOWN"  |"APPROVAL_RATE_PCT"  |
------------------------------------------------------------------------
|22                |7            |15             |31.8                 |
------------------------------------------------------------------------



In [12]:
# Extract negative feedback with messages — these are high-priority eval candidates
# Join observability feedback events to the FEEDBACK_REQUEST_MAPPING table for reliable correlation
session.sql("""
SELECT
    f.TIMESTAMP AS FEEDBACK_TIME,
    f.RECORD_ATTRIBUTES:"snow.ai.observability.user.name"::VARCHAR AS USER_NAME,
    f.VALUE:feedback_message::VARCHAR AS FEEDBACK_MESSAGE,
    f.VALUE:positive::BOOLEAN AS IS_POSITIVE,
    m.USER_QUERY AS ORIGINAL_QUERY,
    m.REQUEST_ID AS AGENT_REQUEST_ID
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
)) f
JOIN FEEDBACK_REQUEST_MAPPING m
    ON f.VALUE:feedback_message::VARCHAR = m.FEEDBACK_MESSAGE
    AND f.VALUE:positive::BOOLEAN = (m.FEEDBACK_SIGNAL = 'positive')
WHERE f.RECORD:name = 'CORTEX_AGENT_FEEDBACK'
  AND f.VALUE:positive::BOOLEAN = FALSE
ORDER BY f.TIMESTAMP DESC
""").show(n=30, max_width=160)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"FEEDBACK_TIME"             |"USER_NAME"  |"FEEDBACK_MESSAGE"                                                                         |"IS_POSITIVE"  |"ORIGINAL_QUERY"                                                                   |"AGENT_REQUEST_ID"                    |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-08-10 19:03:33.508000  |PERICKSON    |Tried to answer using campaign data which is unrelated                                     |False          |How much should we p

In [13]:
# Classify negative feedback into issue categories using CORTEX.COMPLETE
session.sql("""
WITH negative_feedback AS (
    SELECT
        VALUE:feedback_message::VARCHAR AS FEEDBACK_MESSAGE,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
      AND VALUE:positive::BOOLEAN = FALSE
      AND VALUE:feedback_message IS NOT NULL
      AND VALUE:feedback_message != ''
)
SELECT
    FEEDBACK_MESSAGE,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'Classify this user feedback about an AI assistant into exactly one category. '
        || 'Categories: WRONG_ANSWER, MISSING_DETAIL, FORMAT_ISSUE, TOOL_SELECTION_ERROR, SCOPE_ERROR, HALLUCINATION. '
        || 'Return ONLY the category name. '
        || 'Feedback: ' || FEEDBACK_MESSAGE
    )) AS ISSUE_CATEGORY
FROM negative_feedback
ORDER BY TIMESTAMP DESC
""").show(max_width=140)

----------------------------------------------------------------------------------------------------------------
|"FEEDBACK_MESSAGE"                                                                         |"ISSUE_CATEGORY"  |
----------------------------------------------------------------------------------------------------------------
|Didnt structure as headline + bullets + recommendation                                     |FORMAT_ISSUE      |
|Too long and not in executive brief format                                                 |FORMAT_ISSUE      |
|Tried to answer using campaign data which is unrelated                                     |SCOPE_ERROR       |
|Should have declined but tried to write one                                                |SCOPE_ERROR       |
|Couldnt answer - said it doesnt have first-touch data                                      |MISSING_DETAIL    |
|Said it couldnt calculate CPM                                                              |SCO

In [14]:
# Aggregate issue categories to see systemic patterns
session.sql("""
WITH negative_feedback AS (
    SELECT
        VALUE:feedback_message::VARCHAR AS FEEDBACK_MESSAGE
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
      AND VALUE:positive::BOOLEAN = FALSE
      AND VALUE:feedback_message IS NOT NULL
      AND VALUE:feedback_message != ''
),
classified AS (
    SELECT
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Classify this feedback into one category: WRONG_ANSWER, MISSING_DETAIL, FORMAT_ISSUE, TOOL_SELECTION_ERROR, SCOPE_ERROR, HALLUCINATION. Return ONLY the category. Feedback: ' || FEEDBACK_MESSAGE
        )) AS ISSUE_CATEGORY
    FROM negative_feedback
)
SELECT
    ISSUE_CATEGORY,
    COUNT(*) AS COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PCT
FROM classified
GROUP BY ISSUE_CATEGORY
ORDER BY COUNT DESC
""").show()

--------------------------------------
|"ISSUE_CATEGORY"  |"COUNT"  |"PCT"  |
--------------------------------------
|WRONG_ANSWER      |4        |26.7   |
|SCOPE_ERROR       |1        |6.7    |
|MISSING_DETAIL    |8        |53.3   |
|FORMAT_ISSUE      |2        |13.3   |
--------------------------------------



---
## Section 6: Mine Implicit Feedback (Rephrase Detection)

Users who rephrase the same question are implicitly dissatisfied. We detect this with a **two-step approach**:

1. **Similarity filter** — `AI_SIMILARITY` between consecutive user messages in the same thread (threshold > 0.6)
2. **LLM confirmation** — `CORTEX.COMPLETE` determines whether the pair is truly a rephrase (dissatisfaction) vs. a legitimate follow-up question on the same topic

This two-step method avoids false positives. For example:
- *"What was Q4 ROI?"* → *"And how does Q3 compare?"* — similar topic but NOT a rephrase
- *"What was Q4 ROI?"* → *"I meant ROI broken down by channel for Q4 specifically"* — IS a rephrase

Only confirmed rephrases are treated as implicit negative feedback.

In [15]:
# First, let's see what conversation data looks like in the observability table
# Thread structure: first_message_in_thread groups turns, parent_message_id > 0 marks follow-ups
session.sql("""
SELECT
    RECORD_ATTRIBUTES:"snow.ai.observability.agent.first_message_in_thread"::VARCHAR AS THREAD_FIRST_MSG,
    RECORD_ATTRIBUTES:"snow.ai.observability.agent.parent_message_id"::NUMBER AS PARENT_MSG_ID,
    TIMESTAMP,
    RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS USER_MESSAGE,
    RECORD_ATTRIBUTES:"snow.ai.observability.agent.request_id"::VARCHAR AS REQUEST_ID
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
))
WHERE RECORD:name = 'AgentV2RequestResponseInfo'
ORDER BY TIMESTAMP
LIMIT 20
""").show(max_width=140)

-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"THREAD_FIRST_MSG"                               |"PARENT_MSG_ID"  |"TIMESTAMP"                 |"USER_MESSAGE"                                   |"REQUEST_ID"                          |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|What was total spend in 2024?                    |0                |2026-08-10 18:31:45.855000  |What was total spend in 2024?                    |c823c667-539b-448b-bfa2-62c8827ff71d  |
|What was total spend in 2024?                    |0                |2026-08-10 18:31:58.362000  |What was total spend in 2024?                    |6a364034-6154-4154-85e1-ee5deb11dd57  |
|What was total revenue in 2024?                  |0        

In [16]:
# Two-step rephrase detection:
# Step 1: Find follow-up turns (parent_message_id > 0) with high similarity to the first message in their thread
# Step 2: Use CORTEX.COMPLETE to confirm it's actually a rephrase (not just a related follow-up)

session.sql("""
WITH threaded_turns AS (
    SELECT
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.first_message_in_thread"::VARCHAR AS FIRST_MESSAGE,
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS CURRENT_MESSAGE,
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.request_id"::VARCHAR AS REQUEST_ID,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"snow.ai.observability.agent.parent_message_id"::NUMBER > 0
),
-- Step 1: Find candidate pairs with high semantic similarity
similarity_candidates AS (
    SELECT
        FIRST_MESSAGE,
        CURRENT_MESSAGE,
        REQUEST_ID,
        ROUND(VECTOR_COSINE_SIMILARITY(
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
        ), 3) AS SIMILARITY_SCORE
    FROM threaded_turns
    WHERE VECTOR_COSINE_SIMILARITY(
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
    ) > 0.6
)
-- Step 2: LLM confirmation — is it actually a rephrase or a legitimate follow-up?
SELECT
    FIRST_MESSAGE,
    CURRENT_MESSAGE,
    SIMILARITY_SCORE,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'Two consecutive messages from the same user in a conversation with an AI assistant:\n'
        || 'Message 1: ' || FIRST_MESSAGE || '\n'
        || 'Message 2: ' || CURRENT_MESSAGE || '\n\n'
        || 'Is Message 2 a REPHRASE of Message 1 (same underlying question, reworded because the user was unsatisfied), '
        || 'or is it a FOLLOW_UP (a new/different question that builds on or is distinct from Message 1)? '
        || 'Return ONLY one word: REPHRASE or FOLLOW_UP'
    )) AS LLM_CLASSIFICATION
FROM similarity_candidates
ORDER BY SIMILARITY_SCORE DESC
""").show(n=20, max_width=160)


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"FIRST_MESSAGE"                                    |"CURRENT_MESSAGE"                                                                                                                         |"SIMILARITY_SCORE"  |"LLM_CLASSIFICATION"  |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|CPC trend                                          |Show me how cost per click changed month over month for Paid Search throughout 2024                                                       |0.738               |FOLLOW_UP             |
|What is Social Media's share of the total budget?  

In [17]:
# Summary of rephrase detection: how many confirmed rephrases vs follow-ups?
# Threads give us first_message_in_thread and parent_message_id > 0 for follow-up turns
session.sql("""
WITH threaded_turns AS (
    SELECT
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.first_message_in_thread"::VARCHAR AS FIRST_MESSAGE,
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS CURRENT_MESSAGE,
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.parent_message_id"::NUMBER AS PARENT_MSG_ID,
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.request_id"::VARCHAR AS REQUEST_ID,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"snow.ai.observability.agent.parent_message_id"::NUMBER > 0
),
similarity_scored AS (
    SELECT
        *,
        VECTOR_COSINE_SIMILARITY(
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
        ) AS SIM
    FROM threaded_turns
    WHERE VECTOR_COSINE_SIMILARITY(
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
    ) > 0.6
),
confirmed AS (
    SELECT
        *,
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Two messages from same user in a conversation:\nMsg 1: ' || FIRST_MESSAGE || '\nMsg 2: ' || CURRENT_MESSAGE
            || '\nIs Msg 2 a REPHRASE (same question reworded due to dissatisfaction) or FOLLOW_UP (different/new question)? Return ONLY: REPHRASE or FOLLOW_UP'
        )) AS LLM_VERDICT
    FROM similarity_scored
)
SELECT
    COUNT(*) AS HIGH_SIMILARITY_PAIRS,
    SUM(CASE WHEN LLM_VERDICT ILIKE '%REPHRASE%' THEN 1 ELSE 0 END) AS CONFIRMED_REPHRASES,
    SUM(CASE WHEN LLM_VERDICT ILIKE '%FOLLOW%' THEN 1 ELSE 0 END) AS FOLLOW_UPS,
    ROUND(AVG(SIM), 3) AS AVG_SIMILARITY,
    ROUND(SUM(CASE WHEN LLM_VERDICT ILIKE '%REPHRASE%' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS REPHRASE_CONFIRMATION_RATE_PCT
FROM confirmed
""").show()

print("""\nTwo-step detection rationale:
  1. Thread structure (parent_message_id > 0) identifies multi-turn conversations
  2. AI_SIMILARITY > 0.6 between first_message_in_thread and current turn catches candidates
  3. CORTEX.COMPLETE confirms intent match (high precision)
  
  This avoids flagging legitimate follow-ups like:
    'What was Q4 ROI?' -> 'And how does that compare to Q3?'
  which are semantically similar but NOT rephrases.""")


------------------------------------------------------------------------------------------------------------------------
|"HIGH_SIMILARITY_PAIRS"  |"CONFIRMED_REPHRASES"  |"FOLLOW_UPS"  |"AVG_SIMILARITY"  |"REPHRASE_CONFIRMATION_RATE_PCT"  |
------------------------------------------------------------------------------------------------------------------------
|9                        |3                      |6             |0.715             |33.3                              |
------------------------------------------------------------------------------------------------------------------------


Two-step detection rationale:
  1. Thread structure (parent_message_id > 0) identifies multi-turn conversations
  2. AI_SIMILARITY > 0.6 between first_message_in_thread and current turn catches candidates
  3. CORTEX.COMPLETE confirms intent match (high precision)

  This avoids flagging legitimate follow-ups like:
    'What was Q4 ROI?' -> 'And how does that compare to Q3?'
  which are sem

---
## Section 7: Mine Intent Trends

Understanding **what users ask about** reveals coverage gaps and helps ensure the evaluation dataset reflects real usage distribution.

We classify all user queries into intent categories using `CORTEX.COMPLETE`, then look at:
- Which intents are most common?
- Which intents have the highest failure rate?
- Are there intents we're not testing in our eval dataset?

In [18]:
# Classify all user queries into intent categories
session.sql("""
WITH all_requests AS (
    SELECT
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS USER_MESSAGE,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"ai.observability.record_root.input" IS NOT NULL
)
SELECT
    USER_MESSAGE,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'Classify this user question to a marketing AI assistant into exactly one intent category. '
        || 'Categories: ANALYTICS_QUERY (asking for metrics/data/numbers), '
        || 'STRATEGY_LOOKUP (asking about methodology/planning/guidelines), '
        || 'BENCHMARK_COMPARISON (comparing actual vs target), '
        || 'EXECUTIVE_SUMMARY (requesting formatted briefs/summaries), '
        || 'OUT_OF_SCOPE (unrelated to marketing data or strategy). '
        || 'Return ONLY the category name. '
        || 'Question: ' || USER_MESSAGE
    )) AS INTENT
FROM all_requests
ORDER BY TIMESTAMP
""").show(n=30, max_width=140)

------------------------------------------------------------------------------------------------------------
|"USER_MESSAGE"                                                                     |"INTENT"              |
------------------------------------------------------------------------------------------------------------
|Summarize our worst-performing area and what to do about it                        |EXECUTIVE_SUMMARY     |
|What are our competitors spending on paid search?                                  |ANALYTICS_QUERY       |
|What is the target CPC range for Paid Search?                                      |STRATEGY_LOOKUP       |
|Describe our multi-touch attribution approach                                      |STRATEGY_LOOKUP       |
|What are the three inputs for budget allocation?                                   |STRATEGY_LOOKUP       |
|What is the planned Q4 budget increase over Q3?                                    |STRATEGY_LOOKUP       |
|How should year-ov

In [19]:
# Intent distribution — what are users actually asking about?
session.sql("""
WITH all_requests AS (
    SELECT
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS USER_MESSAGE,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"ai.observability.record_root.input" IS NOT NULL
),
classified AS (
    SELECT
        USER_MESSAGE,
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Classify this question into one intent: ANALYTICS_QUERY, STRATEGY_LOOKUP, BENCHMARK_COMPARISON, EXECUTIVE_SUMMARY, OUT_OF_SCOPE. Return ONLY the category. Question: ' || USER_MESSAGE
        )) AS INTENT
    FROM all_requests
)
SELECT
    INTENT,
    COUNT(*) AS QUERY_COUNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PCT_OF_TRAFFIC
FROM classified
GROUP BY INTENT
ORDER BY QUERY_COUNT DESC
""").show()

-----------------------------------------------------------
|"INTENT"              |"QUERY_COUNT"  |"PCT_OF_TRAFFIC"  |
-----------------------------------------------------------
|OUT_OF_SCOPE          |6              |5.0               |
|STRATEGY_LOOKUP       |21             |17.5              |
|EXECUTIVE_SUMMARY     |13             |10.8              |
|ANALYTICS_QUERY       |66             |55.0              |
|BENCHMARK_COMPARISON  |14             |11.7              |
-----------------------------------------------------------



In [20]:
# Cross-reference: which intents have the worst feedback?
# Join via FEEDBACK_REQUEST_MAPPING for reliable feedback-to-query correlation
session.sql("""
WITH classified_feedback AS (
    SELECT
        m.USER_QUERY,
        m.FEEDBACK_SIGNAL,
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Classify into one: ANALYTICS_QUERY, STRATEGY_LOOKUP, BENCHMARK_COMPARISON, EXECUTIVE_SUMMARY, OUT_OF_SCOPE. Return ONLY category. Question: '
            || m.USER_QUERY
        )) AS INTENT
    FROM FEEDBACK_REQUEST_MAPPING m
)
SELECT
    INTENT,
    COUNT(*) AS TOTAL_WITH_FEEDBACK,
    SUM(CASE WHEN FEEDBACK_SIGNAL = 'positive' THEN 1 ELSE 0 END) AS THUMBS_UP,
    SUM(CASE WHEN FEEDBACK_SIGNAL = 'negative' THEN 1 ELSE 0 END) AS THUMBS_DOWN,
    ROUND(SUM(CASE WHEN FEEDBACK_SIGNAL = 'negative' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS FAILURE_RATE_PCT
FROM classified_feedback
GROUP BY INTENT
ORDER BY FAILURE_RATE_PCT DESC
""").show()

---------------------------------------------------------------------------------------------------
|"INTENT"              |"TOTAL_WITH_FEEDBACK"  |"THUMBS_UP"  |"THUMBS_DOWN"  |"FAILURE_RATE_PCT"  |
---------------------------------------------------------------------------------------------------
|ANALYTICS_QUERY       |12                     |3            |9              |75.0                |
|OUT_OF_SCOPE          |1                      |0            |1              |100.0               |
|BENCHMARK_COMPARISON  |3                      |1            |2              |66.7                |
|EXECUTIVE_SUMMARY     |3                      |1            |2              |66.7                |
|STRATEGY_LOOKUP       |3                      |2            |1              |33.3                |
---------------------------------------------------------------------------------------------------



---
## Section 8: Build the Evaluation Dataset from Mined Signals

We now combine all three mining techniques into a single evaluation dataset:

| Source | Priority | Selection Criteria |
|--------|----------|-------------------|
| Explicit negative feedback | **High** | All thumbs-down with feedback messages |
| Implicit rephrase detection | **Medium** | Sessions where similarity > 0.75 |
| Intent coverage sampling | **Low** | Representative sample from each intent |

For each candidate, we use `CORTEX.COMPLETE` to generate a draft ground truth that a human can review.

In [21]:
# Populate MINED_EVAL_CANDIDATES from all three sources

# Source 1: Explicit negative feedback (HIGH priority)
# Uses FEEDBACK_REQUEST_MAPPING for reliable feedback → query correlation
session.sql("""
INSERT INTO MINED_EVAL_CANDIDATES (SOURCE_TYPE, USER_QUERY, FEEDBACK_SIGNAL, FEEDBACK_MESSAGE, SESSION_ID, PRIORITY)
SELECT
    'explicit_negative',
    USER_QUERY,
    'negative',
    FEEDBACK_MESSAGE,
    REQUEST_ID,
    'high'
FROM FEEDBACK_REQUEST_MAPPING
WHERE FEEDBACK_SIGNAL = 'negative'
  AND USER_QUERY IS NOT NULL
""").collect()

print("Source 1 (explicit negative) loaded.")

Source 1 (explicit negative) loaded.


In [22]:
# Source 2: Implicit rephrase detection (MEDIUM priority)
# Only include pairs where BOTH similarity is high AND LLM confirms it's a rephrase
# Use the SECOND message (the clearer, rephrased version) as the eval question
session.sql("""
INSERT INTO MINED_EVAL_CANDIDATES (SOURCE_TYPE, USER_QUERY, FEEDBACK_SIGNAL, FEEDBACK_MESSAGE, SESSION_ID, PRIORITY)
WITH threaded_turns AS (
    SELECT
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.first_message_in_thread"::VARCHAR AS FIRST_MESSAGE,
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS CURRENT_MESSAGE,
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.request_id"::VARCHAR AS REQUEST_ID
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"snow.ai.observability.agent.parent_message_id"::NUMBER > 0
),
similarity_candidates AS (
    SELECT
        FIRST_MESSAGE,
        CURRENT_MESSAGE,
        REQUEST_ID,
        VECTOR_COSINE_SIMILARITY(
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
        ) AS SIM
    FROM threaded_turns
    WHERE VECTOR_COSINE_SIMILARITY(
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', FIRST_MESSAGE)::VECTOR(FLOAT, 768),
        SNOWFLAKE.CORTEX.EMBED_TEXT_768('snowflake-arctic-embed-m-v1.5', CURRENT_MESSAGE)::VECTOR(FLOAT, 768)
    ) > 0.6
),
confirmed_rephrases AS (
    SELECT *
    FROM similarity_candidates
    WHERE TRIM(SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'Two messages from same user in an AI chat:\nMsg 1: ' || FIRST_MESSAGE || '\nMsg 2: ' || CURRENT_MESSAGE
        || '\nIs Msg 2 a REPHRASE (same question reworded because user was unsatisfied with answer) or FOLLOW_UP (genuinely new/different question)? Return ONLY: REPHRASE or FOLLOW_UP'
    )) ILIKE '%REPHRASE%'
)
SELECT
    'implicit_rephrase',
    CURRENT_MESSAGE,
    'rephrase_detected',
    'User rephrased: "' || FIRST_MESSAGE || '" (similarity: ' || ROUND(SIM, 2)::VARCHAR || ', LLM confirmed)',
    REQUEST_ID,
    'medium'
FROM confirmed_rephrases
""").collect()

print("Source 2 (implicit rephrase, LLM-confirmed) loaded.")


Source 2 (implicit rephrase, LLM-confirmed) loaded.


In [23]:
# Source 3: Intent coverage sampling (LOW priority)
# Sample queries from each intent category to ensure broad coverage
session.sql("""
INSERT INTO MINED_EVAL_CANDIDATES (SOURCE_TYPE, USER_QUERY, FEEDBACK_SIGNAL, INTENT_CATEGORY, SESSION_ID, PRIORITY)
WITH classified_requests AS (
    SELECT
        RECORD_ATTRIBUTES:"snow.ai.observability.agent.request_id"::VARCHAR AS REQUEST_ID,
        RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR AS USER_MESSAGE,
        TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Classify into one: ANALYTICS_QUERY, STRATEGY_LOOKUP, BENCHMARK_COMPARISON, EXECUTIVE_SUMMARY, OUT_OF_SCOPE. Return ONLY category. Question: '
            || RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR
        )) AS INTENT,
        ROW_NUMBER() OVER (PARTITION BY
            TRIM(SNOWFLAKE.CORTEX.COMPLETE(
                'mistral-large2',
                'Classify into one: ANALYTICS_QUERY, STRATEGY_LOOKUP, BENCHMARK_COMPARISON, EXECUTIVE_SUMMARY, OUT_OF_SCOPE. Return ONLY category. Question: '
                || RECORD_ATTRIBUTES:"ai.observability.record_root.input"::VARCHAR
            ))
        ORDER BY RANDOM()) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'AgentV2RequestResponseInfo'
      AND RECORD_ATTRIBUTES:"ai.observability.record_root.input" IS NOT NULL
)
SELECT
    'intent_sample',
    USER_MESSAGE,
    'none',
    INTENT,
    REQUEST_ID,
    'low'
FROM classified_requests
WHERE RN <= 3  -- 3 samples per intent category
""").collect()

print("Source 3 (intent sampling) loaded.")

Source 3 (intent sampling) loaded.


In [24]:
# View all mined candidates
session.sql("""
SELECT
    SOURCE_TYPE,
    PRIORITY,
    USER_QUERY,
    FEEDBACK_MESSAGE,
    INTENT_CATEGORY
FROM MINED_EVAL_CANDIDATES
ORDER BY
    CASE PRIORITY WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END,
    SOURCE_TYPE
""").show(n=40, max_width=160)

# Summary
session.sql("""
SELECT
    SOURCE_TYPE,
    PRIORITY,
    COUNT(*) AS CANDIDATES
FROM MINED_EVAL_CANDIDATES
GROUP BY SOURCE_TYPE, PRIORITY
ORDER BY CASE PRIORITY WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END
""").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"SOURCE_TYPE"      |"PRIORITY"  |"USER_QUERY"                                                                                                                              |"FEEDBACK_MESSAGE"                                                                                     |"INTENT_CATEGORY"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|explicit_negative  |high        |What was our ROAS for Display advertising?                              

In [25]:
# Generate draft ground truth for each candidate using CORTEX.COMPLETE
# In production, a human SME would review and refine these
session.sql("""
INSERT INTO OBSERVABILITY_EVAL_DATASET (INPUT_QUERY, GROUND_TRUTH, SOURCE, INTENT, PRIORITY)
WITH generated AS (
    SELECT DISTINCT
        USER_QUERY,
        SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'You are creating ground truth for evaluating a marketing AI assistant. '
            || 'The assistant has access to: (1) campaign spend data for 2024 across Paid Search, Social Media, Email, Display '
            || 'with metrics like spend, revenue, impressions, clicks, conversions; '
            || '(2) strategy documents about attribution methodology, budget allocation, Q4 planning, benchmarks, brand guidelines. '
            || CASE WHEN FEEDBACK_MESSAGE IS NOT NULL AND FEEDBACK_MESSAGE != ''
                THEN 'The user gave this feedback on a prior answer: ' || FEEDBACK_MESSAGE || '. Make sure the ground truth addresses this gap. '
                ELSE '' END
            || 'Generate a JSON object with exactly two keys: '
            || '"ground_truth_output" (string: what a correct answer should say, 1-3 sentences) and '
            || '"expected_tools" (array of strings: "campaign_analytics" and/or "strategy_search"). '
            || 'Return ONLY the raw JSON object, nothing else. No markdown, no backticks, no explanation. '
            || 'Question: ' || USER_QUERY
        ) AS RAW_RESPONSE,
        SOURCE_TYPE,
        COALESCE(INTENT_CATEGORY, TRIM(SNOWFLAKE.CORTEX.COMPLETE(
            'mistral-large2',
            'Classify into one: ANALYTICS_QUERY, STRATEGY_LOOKUP, BENCHMARK_COMPARISON, EXECUTIVE_SUMMARY, OUT_OF_SCOPE. Return ONLY category. Question: ' || USER_QUERY
        ))) AS INTENT,
        PRIORITY
    FROM MINED_EVAL_CANDIDATES
    WHERE USER_QUERY IS NOT NULL
)
SELECT
    USER_QUERY,
    TRY_PARSE_JSON(REGEXP_SUBSTR(RAW_RESPONSE, '\\\\{[\\\\s\\\\S]*\\\\}')),
    SOURCE_TYPE,
    INTENT,
    PRIORITY
FROM generated
WHERE TRY_PARSE_JSON(REGEXP_SUBSTR(RAW_RESPONSE, '\\\\{[\\\\s\\\\S]*\\\\}')) IS NOT NULL
""").collect()

print("Ground truth generated for all candidates.")
session.sql("SELECT COUNT(*) AS EVAL_DATASET_SIZE FROM OBSERVABILITY_EVAL_DATASET").show()

Ground truth generated for all candidates.
-----------------------
|"EVAL_DATASET_SIZE"  |
-----------------------
|66                   |
-----------------------



In [27]:
# Deduplicate the eval dataset — EXECUTE_AI_EVALUATION requires unique INPUT_QUERY values
# Keep the highest-priority row for each query (high > medium > low)
session.sql("""
CREATE OR REPLACE TABLE OBSERVABILITY_EVAL_DATASET AS
SELECT INPUT_QUERY, GROUND_TRUTH, SOURCE, INTENT, PRIORITY FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY INPUT_QUERY
            ORDER BY CASE PRIORITY WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END
        ) AS DEDUP_RN
    FROM OBSERVABILITY_EVAL_DATASET
)
WHERE DEDUP_RN = 1
""").collect()

# Preview the final evaluation dataset
session.sql("""
SELECT
    INPUT_QUERY,
    GROUND_TRUTH:ground_truth_output::VARCHAR AS EXPECTED_ANSWER,
    GROUND_TRUTH:expected_tools AS EXPECTED_TOOLS,
    SOURCE,
    INTENT,
    PRIORITY
FROM OBSERVABILITY_EVAL_DATASET
ORDER BY CASE PRIORITY WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END
LIMIT 20
""").show(max_width=160)

print(f"\nFinal dataset size (deduplicated):")
session.sql("SELECT COUNT(*) AS TOTAL, COUNT(DISTINCT INPUT_QUERY) AS UNIQUE_QUERIES FROM OBSERVABILITY_EVAL_DATASET").show()

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"INPUT_QUERY"                                                                      |"EXPECTED_ANSWER"                                                                                                                                                 |"EXPECTED_TOOLS"         |"SOURCE"           |"INTENT"              |"PRIORITY"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|Tell m

---
## Section 9: Run Evaluation with the Mined Dataset

Now we use Snowflake's built-in `EXECUTE_AI_EVALUATION` to test the CMO_ASSISTANT against the dataset we mined from observability. This closes the flywheel loop: production behavior → evaluation signal → measured improvement.

In [28]:
# Create the evaluation configuration YAML and upload to stage
eval_config = """
dataset:
  dataset_type: "CORTEX AGENT"
  table_name: "CMO_EVAL_LAB.PUBLIC.OBSERVABILITY_EVAL_DATASET"
  dataset_name: "CMO_EVAL_LAB.PUBLIC.OBSERVABILITY_EVAL_DS"
  column_mapping:
    query_text: "INPUT_QUERY"
    ground_truth: "GROUND_TRUTH"

evaluation:
  agent_params:
    agent_name: "CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "Observability-mined evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "OBSERVABILITY_EVAL_DS"

metrics:
  - "answer_correctness"
  - "logical_consistency"
  - name: "tool_selection"
    score_ranges:
      min_score: [0, 0.3]
      median_score: [0.4, 0.6]
      max_score: [0.7, 1.0]
    prompt: |
      Score how well the agent selected the correct tools.
      The ground truth specifies expected_tools (campaign_analytics and/or strategy_search).
      Compare {{ground_truth}} expected tools against {{tool_info}}.
      Score 1.0 if the agent used exactly the correct tool(s).
      Score 0.5 if it used at least one correct tool but also used an unnecessary one.
      Score 0.0 if it used the wrong tool(s) or no tools when tools were expected.
"""

# Write config to stage
session.sql("CREATE OR REPLACE STAGE EVAL_STAGE FILE_FORMAT = (TYPE='CSV' FIELD_DELIMITER=NONE)").collect()

# Use PUT via a temp file
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(eval_config)
    tmp_path = f.name

session.file.put(tmp_path, '@EVAL_STAGE', auto_compress=False, overwrite=True)
print("Evaluation config uploaded to @EVAL_STAGE")
session.sql("LS @EVAL_STAGE").show()

Evaluation config uploaded to @EVAL_STAGE
-----------------------------------------------------------------------------------------------------------
|"name"                       |"size"  |"md5"                             |"last_modified"                |
-----------------------------------------------------------------------------------------------------------
|eval_stage/tmp273ca4tk.yaml  |1168    |a90d1d49a0ca172fe822810770162698  |Mon, 10 Aug 2026 20:01:28 GMT  |
-----------------------------------------------------------------------------------------------------------



In [29]:
# Run the evaluation
import os

# Get the filename that was uploaded
staged_files = session.sql("LS @EVAL_STAGE").collect()
yaml_file = [f['name'] for f in staged_files if f['name'].endswith('.yaml')][0]
yaml_stage_path = f"@EVAL_STAGE/{os.path.basename(yaml_file)}"

run_name = 'observability-mined-v1'

# Drop existing dataset if re-running (the YAML creates it fresh each time)
session.sql("DROP DATASET IF EXISTS OBSERVABILITY_EVAL_DS").collect()

print(f"Starting evaluation run: {run_name}")
print(f"Config: {yaml_stage_path}")

session.sql(f"""
CALL EXECUTE_AI_EVALUATION(
    'START',
    OBJECT_CONSTRUCT('run_name', '{run_name}'),
    '{yaml_stage_path}'
)
""").show()

print("\nEvaluation started. Polling for completion...")

Starting evaluation run: observability-mined-v1
Config: @EVAL_STAGE/tmp273ca4tk.yaml
-----------------------------------
|"RESULT"                         |
-----------------------------------
|Function executed successfully.  |
-----------------------------------


Evaluation started. Polling for completion...


In [30]:
# Poll until evaluation completes
import time

for attempt in range(60):  # max 10 minutes
    status_result = session.sql(f"""
    CALL EXECUTE_AI_EVALUATION(
        'STATUS',
        OBJECT_CONSTRUCT('run_name', '{run_name}'),
        '{yaml_stage_path}'
    )
    """).collect()

    status_str = str(status_result[0][0]) if status_result else ''

    if 'COMPLETED' in status_str.upper() or 'DONE' in status_str.upper():
        print(f"\nEvaluation completed! (attempt {attempt+1})")
        break
    elif 'FAILED' in status_str.upper() or 'ERROR' in status_str.upper():
        print(f"\nEvaluation failed: {status_str}")
        break
    else:
        if attempt % 6 == 0:
            print(f"  Still running... ({attempt*10}s elapsed)")
        time.sleep(10)
else:
    print("Timed out waiting for evaluation. Check status manually.")

  Still running... (0s elapsed)
  Still running... (60s elapsed)
  Still running... (120s elapsed)
  Still running... (180s elapsed)
  Still running... (240s elapsed)
  Still running... (300s elapsed)
  Still running... (360s elapsed)
  Still running... (420s elapsed)
  Still running... (480s elapsed)
  Still running... (540s elapsed)
Timed out waiting for evaluation. Check status manually.


In [31]:
# View aggregate evaluation results
session.sql(f"""
SELECT
    METRIC_NAME,
    ROUND(AVG(EVAL_AGG_SCORE), 3) AS AVG_SCORE,
    ROUND(MIN(EVAL_AGG_SCORE), 3) AS MIN_SCORE,
    ROUND(MAX(EVAL_AGG_SCORE), 3) AS MAX_SCORE,
    COUNT(*) AS NUM_QUESTIONS
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', '{run_name}'
))
GROUP BY METRIC_NAME
ORDER BY AVG_SCORE ASC
""").show()

-----------------------------------------------------------------------------------
|"METRIC_NAME"        |"AVG_SCORE"  |"MIN_SCORE"  |"MAX_SCORE"  |"NUM_QUESTIONS"  |
-----------------------------------------------------------------------------------
|logical_consistency  |0.91         |0.33         |1.0          |37               |
|tool_selection       |0.595        |0.0          |1.0          |37               |
|answer_correctness   |0.549        |0.0          |1.0          |37               |
-----------------------------------------------------------------------------------



In [32]:
# Break down scores by mining source — which source surfaces the hardest questions?
session.sql(f"""
WITH eval_data AS (
    SELECT
        INPUT,
        METRIC_NAME,
        EVAL_AGG_SCORE
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', '{run_name}'
    ))
)
SELECT
    d.SOURCE AS MINING_SOURCE,
    d.PRIORITY,
    e.METRIC_NAME,
    ROUND(AVG(e.EVAL_AGG_SCORE), 3) AS AVG_SCORE,
    COUNT(*) AS N
FROM eval_data e
JOIN OBSERVABILITY_EVAL_DATASET d ON e.INPUT = d.INPUT_QUERY
GROUP BY d.SOURCE, d.PRIORITY, e.METRIC_NAME
ORDER BY d.PRIORITY, d.SOURCE, e.METRIC_NAME
""").show(n=30)

----------------------------------------------------------------------------
|"MINING_SOURCE"    |"PRIORITY"  |"METRIC_NAME"        |"AVG_SCORE"  |"N"  |
----------------------------------------------------------------------------
|implicit_rephrase  |medium      |tool_selection       |0.333        |3    |
|implicit_rephrase  |medium      |logical_consistency  |1.0          |3    |
|explicit_negative  |high        |tool_selection       |0.531        |16   |
|intent_sample      |low         |answer_correctness   |0.648        |18   |
|intent_sample      |low         |tool_selection       |0.694        |18   |
|implicit_rephrase  |medium      |answer_correctness   |0.667        |3    |
|intent_sample      |low         |logical_consistency  |0.907        |18   |
|explicit_negative  |high        |answer_correctness   |0.416        |16   |
|explicit_negative  |high        |logical_consistency  |0.896        |16   |
----------------------------------------------------------------------------

In [33]:
# Identify the worst-performing questions — these are the top improvement opportunities
session.sql(f"""
SELECT
    INPUT AS QUESTION,
    METRIC_NAME,
    ROUND(EVAL_AGG_SCORE, 3) AS SCORE,
    METRIC_CALLS[0]:explanation::VARCHAR AS EXPLANATION
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', '{run_name}'
))
WHERE EVAL_AGG_SCORE < 0.5
ORDER BY EVAL_AGG_SCORE ASC
LIMIT 15
""").show(max_width=160)

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"QUESTION"                                                                                                            |"METRIC_NAME"       |"SCORE"  |"EXPLANATION"                                                                                                                                                     |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|Write me a blog post about digital marketing trends   

---
## Section 10: The Flywheel — Summary & Next Steps

### What We Accomplished

```
┌─────────────────────────────────────────────────────────────────┐
│                    THE OBSERVABILITY FLYWHEEL                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│     ┌──────────┐     ┌──────────┐     ┌──────────┐            │
│     │  DEPLOY  │────▶│ OBSERVE  │────▶│   MINE   │            │
│     └──────────┘     └──────────┘     └──────────┘            │
│          ▲                                  │                   │
│          │                                  ▼                   │
│     ┌──────────┐                     ┌──────────┐             │
│     │ IMPROVE  │◀────────────────────│ EVALUATE │             │
│     └──────────┘                     └──────────┘             │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│  OBSERVE: AI_OBSERVABILITY_EVENTS captures every interaction    │
│  MINE:    Extract signal from feedback, rephrases, intents      │
│  EVALUATE: Test against mined dataset with LLM-as-judge        │
│  IMPROVE: Fix orchestration, tools, instructions from findings  │
│  DEPLOY:  Version, evaluate, promote with confidence            │
└─────────────────────────────────────────────────────────────────┘
```

| Step | What we did |
|------|-------------|
| **Simulate** | Sent ~100 queries + 10 multi-turn conversations to CMO_ASSISTANT |
| **Explicit feedback** | Submitted thumbs-up/down via Feedback REST API |
| **Implicit feedback** | Detected rephrasing via `AI_SIMILARITY` on consecutive turns |
| **Intent mining** | Classified all queries with `CORTEX.COMPLETE` |
| **Cross-reference** | Found which intents have the worst feedback rates |
| **Dataset assembly** | Combined all signals into `OBSERVABILITY_EVAL_DATASET` |
| **Ground truth** | Auto-generated draft ground truth with `CORTEX.COMPLETE` |
| **Evaluation** | Ran `EXECUTE_AI_EVALUATION` against the mined dataset |
| **Analysis** | Identified worst performers by source and intent |

### Key Insight

The observability-mined dataset tests the agent on its **actual weaknesses** — not hypothetical ones. Negative feedback and rephrase detection surface the real failure modes that hurt user trust.

### Next Steps

1. **Human review** — Have SMEs review and refine the auto-generated ground truth
2. **Fix the agent** — Improve orchestration instructions based on failure patterns (MISSING_DETAIL, TOOL_SELECTION_ERROR)
3. **Re-evaluate** — Run the same dataset against the improved agent version
4. **Automate** — Schedule periodic mining with a Snowflake Task to keep the eval dataset fresh
5. **Monitor** — Track failure rate by intent over time to catch regressions early

In [34]:
# Final summary statistics
print("=" * 60)
print("OBSERVABILITY MINING SUMMARY")
print("=" * 60)

stats = session.sql("""
SELECT
    (SELECT COUNT(*) FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    )) WHERE RECORD:name = 'CORTEX_AGENT_REQUEST') AS TOTAL_REQUESTS,
    (SELECT COUNT(*) FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    )) WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK') AS TOTAL_FEEDBACK,
    (SELECT COUNT(*) FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT'
    )) WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK' AND VALUE:positive::BOOLEAN = FALSE) AS NEGATIVE_FEEDBACK,
    (SELECT COUNT(*) FROM MINED_EVAL_CANDIDATES) AS TOTAL_CANDIDATES,
    (SELECT COUNT(*) FROM OBSERVABILITY_EVAL_DATASET) AS EVAL_DATASET_SIZE
""").collect()[0]

print(f"\n  Total agent requests:       {stats['TOTAL_REQUESTS']}")
print(f"  Total feedback events:      {stats['TOTAL_FEEDBACK']}")
print(f"  Negative feedback:          {stats['NEGATIVE_FEEDBACK']}")
print(f"  Mined eval candidates:      {stats['TOTAL_CANDIDATES']}")
print(f"  Final eval dataset size:    {stats['EVAL_DATASET_SIZE']}")
print(f"\n{'=' * 60}")
print("The flywheel is turning. Every interaction makes the agent better.")

OBSERVABILITY MINING SUMMARY

  Total agent requests:       157
  Total feedback events:      22
  Negative feedback:          15
  Mined eval candidates:      65
  Final eval dataset size:    37

The flywheel is turning. Every interaction makes the agent better.


In [35]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP TABLE IF EXISTS MINED_EVAL_CANDIDATES").collect()
# session.sql("DROP TABLE IF EXISTS OBSERVABILITY_EVAL_DATASET").collect()
# print("Cleaned up mining tables.")